In [ ]:
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
OUTPUT_DIR = "../model_evaluation/summary"

PERTURBATION_SCENARIOS = [
    "gaussian_noise",
    "blur",
    "brightness_shift",
]
SEVERITY_LABELS = {
    ("clean", "default"): "Clean",
    ("gaussian_noise", "severity_level_1"): "$\\sigma_1=0.03$",
    ("gaussian_noise", "severity_level_2"): "$\\sigma_2=0.1$",
    ("gaussian_noise", "severity_level_3"): "$\\sigma_3=0.3$",
    ("blur", "severity_level_1"): "$k_1=3$",
    ("blur", "severity_level_2"): "$k_2=5$",
    ("blur", "severity_level_3"): "$k_3=7$",
    ("brightness_shift", "severity_level_1"): "$\\beta_1=-0.3$",
    ("brightness_shift", "severity_level_2"): "$\\beta_2=0.05$",
    ("brightness_shift", "severity_level_3"): "$\\beta_3=0.3$",
}
CHECKPOINT_TO_STEP_MAPPING = {
    "000000": 0,
    "004000": 4000,
    "008000": 8000,
    "012000": 12000,
    "016000": 16000,
    "020000": 20000,
    "024000": 24000,
    "last": 25650,
}

In [ ]:
success_rates = json.load(open(f"{OUTPUT_DIR}/success_rates.json", "r"))

## Success rate

In [ ]:
def get_success_rate_dataframe(scenario: str, severity_level: str) -> pd.DataFrame:

    df = pd.DataFrame(success_rates[scenario][severity_level])
    df.index = [CHECKPOINT_TO_STEP_MAPPING[ckpt] / 10000 for ckpt in df.index]

    max_success_rates = df.max(axis=1)
    mean_success_rates = df.mean(axis=1)
    min_success_rates = df.min(axis=1)

    df["max_success_rate"] = max_success_rates
    df["mean_success_rate"] = mean_success_rates
    df["min_success_rate"] = min_success_rates

    return df


def draw_success_rate(ax, df: pd.DataFrame, scenario: str, severity_level: str) -> None:

    sns.lineplot(
        data=df["mean_success_rate"] * 100,
        ax=ax,
        label=SEVERITY_LABELS[(scenario, severity_level)],
    )
    ax.fill_between(
        df.index,
        df["min_success_rate"] * 100,
        df["max_success_rate"] * 100,
        alpha=0.2,
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

for i, scenario in enumerate(PERTURBATION_SCENARIOS):
    ax = axes[i]

    df_clean = get_success_rate_dataframe("clean", "default")
    draw_success_rate(ax, df_clean, "clean", "default")

    for severity_level in success_rates[scenario]:
        df_case = get_success_rate_dataframe(scenario, severity_level)
        draw_success_rate(ax, df_case, scenario, severity_level)

    ax.grid()
    ax.set_ylabel("")
    ax.set_xlabel("Training steps ($\\times 10^4$)")
    ax.set_title(scenario.replace("_", " ").title())

axes[0].set_ylabel("Success Rate (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/success_rate.pdf")
plt.savefig(f"{OUTPUT_DIR}/success_rate.png")

## Success rate degradation

In [ ]:
def draw_success_rate_degradation(ax, df_clean: pd.DataFrame, df_case: pd.DataFrame, scenario: str, severity_level: str) -> None:

    sns.lineplot(
        data=(df_clean["mean_success_rate"] - df_case["mean_success_rate"]).clip(
            lower=0
        )
        * 100,
        ax=ax,
        label=SEVERITY_LABELS[(scenario, severity_level)],
    )
    ax.fill_between(
        df_case.index,
        (df_clean["min_success_rate"] - df_case["max_success_rate"]).clip(lower=0)
        * 100,
        (df_clean["max_success_rate"] - df_case["min_success_rate"]).clip(lower=0)
        * 100,
        alpha=0.2,
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

for i, scenario in enumerate(PERTURBATION_SCENARIOS):
    ax = axes[i]

    df_clean = get_success_rate_dataframe("clean", "default")

    for severity_level in success_rates[scenario]:
        df_case = get_success_rate_dataframe(scenario, severity_level)
        draw_success_rate_degradation(ax, df_clean, df_case, scenario, severity_level)

    ax.grid()
    ax.set_ylabel("")
    ax.set_ylim(0, 35)
    ax.set_xlabel("Training steps ($\\times 10^4$)")
    ax.set_title(scenario.replace("_", " ").title())

axes[0].set_ylabel("Success Rate Degradation (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/success_rate_degradation.pdf")
plt.savefig(f"{OUTPUT_DIR}/success_rate_degradation.png")

## Final-checkpoint barchart

In [ ]:
df_final_ckpt = pd.DataFrame()

df_case = get_success_rate_dataframe("clean", "default")
df_final_ckpt[SEVERITY_LABELS[("clean", "default")]] = df_case.loc[
    CHECKPOINT_TO_STEP_MAPPING["last"] / 10000,
    [
        "max_success_rate", "mean_success_rate", "min_success_rate"
    ]
]

for scenario in PERTURBATION_SCENARIOS:
    for severity_level in success_rates[scenario]:
        df_case = get_success_rate_dataframe(scenario, severity_level)
        key = SEVERITY_LABELS[(scenario, severity_level)]
        df_final_ckpt[key] = df_case.loc[
            CHECKPOINT_TO_STEP_MAPPING["last"] / 10000,
            [
                "max_success_rate", "mean_success_rate", "min_success_rate"
            ]
        ]
df_final_ckpt = df_final_ckpt.T * 100

df_final_ckpt["mean_success_rate_degradation"] = (
    df_final_ckpt.loc["Clean", "mean_success_rate"] - df_final_ckpt["mean_success_rate"]
).clip(lower=0)
df_final_ckpt["min_success_rate_degradation"] = (
    df_final_ckpt.loc["Clean", "min_success_rate"] - df_final_ckpt["max_success_rate"]
).clip(lower=0)
df_final_ckpt["max_success_rate_degradation"] = (
    df_final_ckpt.loc["Clean", "max_success_rate"] - df_final_ckpt["min_success_rate"]
).clip(lower=0)
df_final_ckpt.loc["Clean", "max_success_rate_degradation"] = 0

df_final_ckpt

In [ ]:
def draw_final_checkpoint_property_bar_chart(
    df_final_checkpoint: pd.DataFrame,
    mean_col: str,
    min_col: str,
    max_col: str,
    ylabel: str,
):

    fig, ax = plt.subplots(figsize=(9, 3.5))

    x = np.arange(len(df_final_checkpoint))
    y = df_final_checkpoint[mean_col]

    yerr = [
        y - df_final_checkpoint[min_col],
        df_final_checkpoint[max_col] - y,
    ]

    colors = [
        "tab:green",   # Clean
        "tab:blue",    # Gaussian noise severity 1
        "tab:blue",    # Gaussian noise severity 2
        "tab:blue",    # Gaussian noise severity 3
        "tab:orange",  # Blur severity 1
        "tab:orange",  # Blur severity 2
        "tab:orange",  # Blur severity 3
        "tab:purple",  # Brightness shift severity 1
        "tab:purple",  # Brightness shift severity 2
        "tab:purple",  # Brightness shift severity 3
    ]

    ax.bar(
        x,
        y,
        yerr=yerr,
        capsize=4,
        color=colors[-len(df_final_checkpoint):],
        edgecolor="black",
        linewidth=0.8,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(df_final_checkpoint.index, rotation=35, ha="right")

    ax.set_xlabel("Visual Condition")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.4)

    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, color="tab:blue", label="Gaussian Noise"),
        plt.Rectangle((0, 0), 1, 1, color="tab:orange", label="Blur"),
        plt.Rectangle((0, 0), 1, 1, color="tab:purple", label="Brightness Shift"),
    ]

    if len(df_final_checkpoint) == len(colors):
        legend_handles = [plt.Rectangle((0, 0), 1, 1, color="tab:green", label="Clean")] + legend_handles

    # Leave space on the right for the legend
    fig.subplots_adjust(right=0.78)

    # Place legend outside the plot
    fig.legend(
        handles=legend_handles,
        title="Scenario",
        frameon=True,
        loc="center right",
        bbox_to_anchor=(0.98, 0.5),
    )
    return fig, ax

In [ ]:
fig, ax = draw_final_checkpoint_property_bar_chart(
    df_final_checkpoint=df_final_ckpt,
    mean_col="mean_success_rate",
    min_col="min_success_rate",
    max_col="max_success_rate",
    ylabel="Success Rate (%)"
)

plt.savefig(
    f"{OUTPUT_DIR}/final_checkpoint_success_rate.pdf",
    bbox_inches="tight",
    pad_inches=0.05,
)
plt.savefig(
    f"{OUTPUT_DIR}/final_checkpoint_success_rate.png",
    bbox_inches="tight",
    pad_inches=0.05,
)

In [ ]:
fig, ax = draw_final_checkpoint_property_bar_chart(
    df_final_checkpoint=df_final_ckpt.loc[df_final_ckpt.index != "Clean"],
    mean_col="mean_success_rate_degradation",
    min_col="min_success_rate_degradation",
    max_col="max_success_rate_degradation",
    ylabel="Success Rate Degradation (%)"
)

plt.savefig(
    f"{OUTPUT_DIR}/final_checkpoint_success_rate_degradation.pdf",
    bbox_inches="tight",
    pad_inches=0.05,
)
plt.savefig(
    f"{OUTPUT_DIR}/final_checkpoint_success_rate_degradation.png",
    bbox_inches="tight",
    pad_inches=0.05,
)

## Final-checkpoint summary table

In [ ]:
df_final_ckpt["Success Rate (%)"] = df_final_ckpt.apply(
    lambda r: (
        f"${r['mean_success_rate']:.2f}"
        f"^{{+{(r['max_success_rate'] - r['mean_success_rate']):.2f}}}"
        f"_{{-{(r['mean_success_rate'] - r['min_success_rate']):.2f}}}$"
    ),
    axis=1
)

df_final_ckpt["Success Rate Degradation (%)"] = df_final_ckpt.apply(
    lambda r: (
        f"${r['mean_success_rate_degradation']:.2f}"
        f"^{{+{(r['max_success_rate_degradation'] - r['mean_success_rate_degradation']):.2f}}}"
        f"_{{-{(r['mean_success_rate_degradation'] - r['min_success_rate_degradation']):.2f}}}$"
    ),
    axis=1
)

print(df_final_ckpt[["Success Rate (%)", "Success Rate Degradation (%)"]].to_latex())